# 07.00 — Target Design

Review the formal Forecasting and Peak-Risk target definitions before constructing target artifacts.

In [1]:
# Import libraries
from pathlib import Path
import sys
import yaml

In [2]:
# Define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'target_definition.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk/configs/target_definition.yaml')

In [3]:
# Import module to manage the Target Definition process
from src.ontario_peak_risk.target_definition.common import (
    load_target_config,
    load_feature_dataset,
    ensure_directories,
)

In [4]:
# Configure the Target Definition process
CONFIG, _ = load_target_config(CONFIG_PATH)
OUTPUT_DIR, REPORTS_DIR, DOCS_DIR = ensure_directories(CONFIG, PROJECT_ROOT)
feature_dataset = load_feature_dataset(CONFIG, PROJECT_ROOT)
feature_dataset.shape

(262944, 100)

In [5]:
# Display the target definition configuration
with CONFIG_PATH.open('r', encoding='utf-8') as file:
    target_config = yaml.safe_load(file)
target_config['target_definition']

{'key_columns': ['fsa', 'timestamp'],
 'forecasting': {'target_column': 'total_consumption_kwh',
  'horizon_hours': 24,
  'target_prefix': 'target_h',
  'require_complete_horizon_for_modeling': True},
 'peak_risk': {'target_column': 'total_consumption_kwh',
  'percentile_threshold': 0.975,
  'grouping_columns': ['fsa', 'season'],
  'diagnostic_time_column': 'year',
  'minimum_training_years': 1},
 'validation': {'require_unique_key': True,
  'require_hourly_alignment': True,
  'reject_duplicate_target_origins': True,
  'allow_incomplete_last_horizon_rows': True}}

### Design principle

- The forecasting labels may be materialized statically because they are supervised-learning outcomes. 
- Peak-Risk thresholds must be fitted using training data only and therefore cannot be created once globally for final model evaluation.